# 1. LIBRARIES 

In [1]:
from pathlib import Path
import json

import torch
import pandas as pd

import cv2
cv2.setNumThreads(0)

from ultralytics import YOLO

In [ ]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)

if DEVICE == 0:
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: 0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


# 2. LOAD PRE-TRAINED MODEL YOLO11N-SEG

In [3]:
MODEL_NAME = "yolo11n-seg.pt"
model = YOLO(MODEL_NAME)

model.info()


YOLO11n-seg summary: 203 layers, 2,876,848 parameters, 0 gradients, 10.0 GFLOPs


(203, 2876848, 0, 9.9593344)

# 3. INSPECT MODEL LAYER 

In [4]:
for i, layer in enumerate(model.model.model):
    print(
        f"{i:02d} | "
        f"{layer.__class__.__name__}"
    )

00 | Conv
01 | Conv
02 | C3k2
03 | Conv
04 | C3k2
05 | Conv
06 | C3k2
07 | Conv
08 | C3k2
09 | SPPF
10 | C2PSA
11 | Upsample
12 | Concat
13 | C3k2
14 | Upsample
15 | Concat
16 | C3k2
17 | Conv
18 | Concat
19 | C3k2
20 | Conv
21 | Concat
22 | C3k2
23 | Segment


In [5]:
print("=== MODEL PARAMETER DIAGNOSTIC ===\n")

total = 0
trainable = 0
frozen = 0

for name, param in model.model.named_parameters():
    total += param.numel()

    if param.requires_grad:
        trainable += param.numel()
    else:
        frozen += param.numel()

    print(
        f"{name:70s} | "
        f"requires_grad={param.requires_grad} | "
        f"shape={tuple(param.shape)}"
    )

print("\n=== SUMMARY ===")
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters:    {frozen:,}")

=== MODEL PARAMETER DIAGNOSTIC ===

model.0.conv.weight                                                    | requires_grad=False | shape=(16, 3, 3, 3)
model.0.bn.weight                                                      | requires_grad=False | shape=(16,)
model.0.bn.bias                                                        | requires_grad=False | shape=(16,)
model.1.conv.weight                                                    | requires_grad=False | shape=(32, 16, 3, 3)
model.1.bn.weight                                                      | requires_grad=False | shape=(32,)
model.1.bn.bias                                                        | requires_grad=False | shape=(32,)
model.2.cv1.conv.weight                                                | requires_grad=False | shape=(32, 32, 1, 1)
model.2.cv1.bn.weight                                                  | requires_grad=False | shape=(32,)
model.2.cv1.bn.bias                                                    | requires_

In [6]:

for param in model.model.parameters():
    param.requires_grad = True

print("All parameters restored to trainable.")

All parameters restored to trainable.


In [7]:
trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in model.model.parameters()
    if not p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

Total parameters:     2,876,848
Trainable parameters: 2,876,848
Frozen parameters:    0


# 4. FREEZE BACKBONE LAYER 

In [8]:
BACKBONE_END = 10
FREEZE_LAYERS = BACKBONE_END + 1

for i, layer in enumerate(model.model.model):
    if i <= BACKBONE_END:
        for param in layer.parameters():
            param.requires_grad = False

print("Backbone layers 00–10 frozen.")

Backbone layers 00–10 frozen.


In [9]:
for i, layer in enumerate(model.model.model):
    params = list(layer.parameters())

    if len(params) == 0:
        status = "NO PARAMETERS"
    else:
        trainable = any(p.requires_grad for p in params)
        status = "TRAINABLE" if trainable else "FROZEN"

    print(
        f"{i:02d} | "
        f"{layer.__class__.__name__:12s} | "
        f"{status}"
    )

00 | Conv         | FROZEN
01 | Conv         | FROZEN
02 | C3k2         | FROZEN
03 | Conv         | FROZEN
04 | C3k2         | FROZEN
05 | Conv         | FROZEN
06 | C3k2         | FROZEN
07 | Conv         | FROZEN
08 | C3k2         | FROZEN
09 | SPPF         | FROZEN
10 | C2PSA        | FROZEN
11 | Upsample     | NO PARAMETERS
12 | Concat       | NO PARAMETERS
13 | C3k2         | TRAINABLE
14 | Upsample     | NO PARAMETERS
15 | Concat       | NO PARAMETERS
16 | C3k2         | TRAINABLE
17 | Conv         | TRAINABLE
18 | Concat       | NO PARAMETERS
19 | C3k2         | TRAINABLE
20 | Conv         | TRAINABLE
21 | Concat       | NO PARAMETERS
22 | C3k2         | TRAINABLE
23 | Segment      | TRAINABLE


# 5. STAGE 1 FINE TUNING CONFIGURATION 

In [10]:
PROJECT_ROOT = Path("..").resolve()
DATASET_VERSION = "v4"
STAGE = "stage1"

In [11]:
DATASET_DIR = PROJECT_ROOT / "datasets" / DATASET_VERSION
DATA_YAML = DATASET_DIR / "data.yaml"

In [12]:
RUNS_DIR = PROJECT_ROOT / "runs"

EXPERIMENT_DIR = (
    RUNS_DIR
    / DATASET_VERSION
    / "segmentation"
    / "stage1"
)

EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [13]:
STAGE1_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 50,
    "imgsz": 640,
    "batch": 8,
    "device": DEVICE,
    "freeze": FREEZE_LAYERS,
    "workers": 2,          
    "save_period": 5,       

    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.0005,

    "warmup_epochs": 3,
    "patience": 15,

    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
    "degrees": 5.0, "translate": 0.1, "scale": 0.5, "fliplr": 0.5,
    "copy_paste": 0.4,
    "copy_paste_mode": "mixup",
    "erasing": 0.1,
    "mask_ratio": 2,

    "amp": True,
    "seed": 42,
}

In [14]:


print("Model:", "YOLO11n-Seg")
print("Dataset:", DATA_YAML)
print("Device:", DEVICE)

print("\nParameters:")

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print(f"Total:      {total_params:,}")
print(f"Trainable:  {trainable_params:,}")
print(f"Frozen:     {frozen_params:,}")

print("\nExpected:")
print("Backbone 00–10 → FROZEN")
print("Neck 11–22     → TRAINABLE")
print("Head 23        → TRAINABLE")

Model: YOLO11n-Seg
Dataset: D:\PREP_INTERN\nutrivision_pro\datasets\v4\data.yaml
Device: 0

Parameters:
Total:      2,876,848
Trainable:  1,511,376
Frozen:     1,365,472

Expected:
Backbone 00–10 → FROZEN
Neck 11–22     → TRAINABLE
Head 23        → TRAINABLE


# 6. FINE TUNING 

In [15]:
LAST_CKPT = EXPERIMENT_DIR / "weights" / "last.pt"
resume_arg = str(LAST_CKPT) if LAST_CKPT.exists() else False

stage1_results = model.train(
    **STAGE1_CONFIG,
    resume=resume_arg,     # sekarang path string spesifik, bukan True/False
    project=str(EXPERIMENT_DIR.parent),
    name=EXPERIMENT_DIR.name,
    exist_ok=True,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)  
print(f"Run saved in : {RUN_DIR}")
print(f"Best checkpoint : {RUN_DIR / 'weights' / 'best.pt'}")

New https://pypi.org/project/ultralytics/8.4.152 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.4, copy_paste_mode=mixup, cos_lr=False, cutmix=0.0, data=D:\PREP_INTERN\nutrivision_pro\datasets\v4\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.1, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=11, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=2, max_d

d:\PREP_INTERN\nutrivision_pro\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-15 12:07:28,973	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-09-15 12:07:32,788	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Overriding model.yaml nc=80 with nc=15

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytic

d:\PREP_INTERN\nutrivision_pro\venv\Lib\site-packages\ray\train\_internal\session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      7.13G       1.05      2.502      2.707      1.217          0         62        640: 100% ━━━━━━━━━━━━ 97/97 2.2it/s 43.4s0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.9it/s 1.4s0.6s
                   all         55        556      0.539      0.173      0.165      0.111      0.516      0.137      0.122     0.0591

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/50      7.31G      1.053      2.441      2.475      1.218          0        103        640: 100% ━━━━━━━━━━━━ 97/97 2.4it/s 40.4s0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.7it/s 1.5s0.6s
                   all         55        556      0.416      0.214      0.179      0.117      0.407      0.203      0.162      0.093

In [16]:
print("=== Verification after freeze the layer ===")
trained = YOLO(RUN_DIR / "weights" / "last.pt")  
for i, layer in enumerate(trained.model.model):
    params = list(layer.parameters())
    status = "NO PARAMS" if not params else ("TRAINABLE" if any(p.requires_grad for p in params) else "FROZEN")
    print(f"{i:02d} | {layer.__class__.__name__:12s} | {status}")

=== Verification after freeze the layer ===
00 | Conv         | FROZEN
01 | Conv         | FROZEN
02 | C3k2         | FROZEN
03 | Conv         | FROZEN
04 | C3k2         | FROZEN
05 | Conv         | FROZEN
06 | C3k2         | FROZEN
07 | Conv         | FROZEN
08 | C3k2         | FROZEN
09 | SPPF         | FROZEN
10 | C2PSA        | FROZEN
11 | Upsample     | NO PARAMS
12 | Concat       | NO PARAMS
13 | C3k2         | FROZEN
14 | Upsample     | NO PARAMS
15 | Concat       | NO PARAMS
16 | C3k2         | FROZEN
17 | Conv         | FROZEN
18 | Concat       | NO PARAMS
19 | C3k2         | FROZEN
20 | Conv         | FROZEN
21 | Concat       | NO PARAMS
22 | C3k2         | FROZEN
23 | Segment      | FROZEN


In [17]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.14.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB


# 7. VALIDATION FINE TUNING STAGE 1 

## 7.1. Load best.pt

In [18]:
from ultralytics import YOLO

# Best model from current experiment
BEST_MODEL_PATH = EXPERIMENT_DIR / "weights" / "best.pt"

# Check model exists
assert BEST_MODEL_PATH.exists(), (
    f"Best model not found:\n{BEST_MODEL_PATH}"
)

# Load model
model = YOLO(str(BEST_MODEL_PATH))

print("Best model loaded.")
print("Model path :", BEST_MODEL_PATH)
print("Dataset    :", DATASET_VERSION)
print("Stage      :", STAGE)

Best model loaded.
Model path : D:\PREP_INTERN\nutrivision_pro\runs\v4\segmentation\stage1\weights\best.pt
Dataset    : v4
Stage      : stage1


In [19]:
results = model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    plots=True,
    verbose=True
)

Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)


YOLO11n-seg summary (fused): 113 layers, 2,837,493 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 296.668.2 MB/s, size: 55.9 KB)
val: Scanning D:\PREP_INTERN\nutrivision_pro\datasets\v4\valid\labels.cache... 55 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 55/55 12.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 6.3it/s 2.2s0.1s
                   all         55        556      0.487      0.424      0.425      0.328      0.467      0.425       0.41      0.295
                  beef          7         20      0.551        0.3      0.294       0.22      0.414        0.3      0.285      0.207
               chicken         20         91      0.272     0.0879      0.176      0.118      0.295      0.101      0.166      0.112
                   egg         13         14      0.238      0.643      0.463      0.411      0.224      

# 8. PER CLASS PEFORMANCE

In [20]:
import pandas as pd
import numpy as np


class_names = model.names
class_indices = results.seg.ap_class_index

class_metrics = pd.DataFrame({
    "class_id": class_indices,
    "class_name": [
        class_names[int(i)]
        for i in class_indices
    ],

    "precision": results.seg.p,
    "recall": results.seg.r,
    "mAP50": results.seg.ap50,
    "mAP50-95": results.seg.ap,
})


class_metrics["f1"] = (
    2
    * class_metrics["precision"]
    * class_metrics["recall"]
    / (
        class_metrics["precision"]
        + class_metrics["recall"]
    ).replace(0, np.nan)
)


class_metrics = class_metrics.round(4)

class_metrics

,class_id,class_name,precision,recall,mAP50,mAP50-95,f1
0,0,beef,0.4139,0.3000,0.2847,0.2069,0.3479
1,1,chicken,0.2951,0.1013,0.1657,0.1122,0.1508
2,2,egg,0.2245,0.6429,0.4628,0.4317,0.3327
3,3,fish,0.6086,0.5000,0.4978,0.1911,0.5490
4,4,fruit,0.7122,0.8095,0.8026,0.6323,0.7577
5,5,noodles,0.5554,0.4444,0.4228,0.2992,0.4938
6,6,other_carbs,0.0000,0.0000,0.0214,0.0121,NaN
7,7,pork,0.4559,0.3019,0.2939,0.1571,0.3632
8,8,rice,0.7180,0.7045,0.7252,0.5995,0.7112
9,9,sambal,0.5268,0.6667,0.7888,0.6602,0.5886


In [21]:
CSV_PATH = (
    PROJECT_ROOT
    / "runs"
    / DATASET_VERSION
    / "per_class_segmentation_metrics.csv"
)

CSV_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

class_metrics.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved to:")
print(CSV_PATH.resolve())

CSV saved to:
D:\PREP_INTERN\nutrivision_pro\runs\v4\per_class_segmentation_metrics.csv


In [22]:
# Cari threshold confidence yang memaksimalkan F1 per kelas dari kurva PR,
# bukan pakai satu angka default 0.25 untuk semua kelas
f1_curve = results.seg.curves_results[1]   # index 1 = "F1-Confidence(M)"
conf_values, f1_matrix = f1_curve[0], f1_curve[1]

optimal_thresholds = {}
for i, class_id in enumerate(class_indices):
    class_name = class_names[int(class_id)]
    best_idx = f1_matrix[i].argmax()
    optimal_thresholds[class_name] = round(float(conf_values[best_idx]), 3)

print("=== Optimal confidence threshold per kelas ===")
for name, thr in optimal_thresholds.items():
    print(f"{name:15s} : {thr}")

THRESHOLD_JSON = EXPERIMENT_DIR / "optimal_class_thresholds.json"
with open(THRESHOLD_JSON, "w") as f:
    json.dump(optimal_thresholds, f, indent=2)
print(f"\nDisimpan ke: {THRESHOLD_JSON}")

=== Optimal confidence threshold per kelas ===
beef            : 0.342
chicken         : 0.038
egg             : 0.718
fish            : 0.636
fruit           : 0.309
noodles         : 0.639
other_carbs     : 0.107
pork            : 0.234
rice            : 0.643
sambal          : 0.868
shrimp          : 0.324
squid           : 0.012
tempeh          : 0.811
tofu            : 0.081
vegetable       : 0.259

Disimpan ke: D:\PREP_INTERN\nutrivision_pro\runs\v4\segmentation\stage1\optimal_class_thresholds.json


In [27]:

def evaluate_dataset_per_image_class_presence(
    model,
    dataset_dir,
    splits,
    conf_thresholds,
    default_conf=0.25,
):
    class_names_list = list(model.names.values())

    results_log = []

    for split in splits:
        images_dir = dataset_dir / split / "images"
        labels_dir = dataset_dir / split / "labels"

        if not images_dir.exists():
            print(f"[WARNING] Images directory tidak ditemukan: {images_dir}")
            continue

        if not labels_dir.exists():
            print(f"[WARNING] Labels directory tidak ditemukan: {labels_dir}")
            continue

        image_paths = sorted(images_dir.glob("*.jpg"))

        print(f"\nEvaluating {split.upper()}: {len(image_paths)} images")

        for image_path in image_paths:

            label_path = labels_dir / f"{image_path.stem}.txt"

            # =========================================================
            # 1. GROUND TRUTH CLASS PRESENCE
            # =========================================================
            gt_classes = set()

            if label_path.exists():
                for line in label_path.read_text().splitlines():

                    if line.strip():
                        class_id = int(line.split()[0])
                        gt_classes.add(class_id)

            # =========================================================
            # 2. MODEL PREDICTION
            # =========================================================
            # Gunakan confidence rendah terlebih dahulu,
            # kemudian filter manual berdasarkan optimal threshold
            # masing-masing class.
            pred = model.predict(
                image_path,
                conf=0.05,
                verbose=False
            )[0]

            pred_classes = set()

            if pred.boxes is not None:

                for cls_idx, conf in zip(
                    pred.boxes.cls,
                    pred.boxes.conf
                ):

                    cls_idx = int(cls_idx)
                    cname = class_names_list[cls_idx]

                    threshold = conf_thresholds.get(
                        cname,
                        default_conf
                    )

                    if float(conf) >= threshold:
                        pred_classes.add(cls_idx)

            # =========================================================
            # 3. COMPARE GT VS PREDICTION
            # =========================================================
            false_negative = gt_classes - pred_classes
            false_positive = pred_classes - gt_classes

            correctly_identified = gt_classes & pred_classes

            # =========================================================
            # 4. LOG RESULT
            # =========================================================
            results_log.append({

                "split": split,

                "image": image_path.name,

                "gt_classes": len(gt_classes),

                "correctly_identified": len(correctly_identified),

                "missed_classes": len(false_negative),

                "spurious_classes": len(false_positive),

                "missed_class_names": ",".join(
                    class_names_list[c]
                    for c in sorted(false_negative)
                ),

                "spurious_class_names": ",".join(
                    class_names_list[c]
                    for c in sorted(false_positive)
                ),

                "image_perfect": (
                    len(false_negative) == 0
                    and len(false_positive) == 0
                ),
            })

    # =============================================================
    # 5. CREATE DATAFRAME
    # =============================================================
    df = pd.DataFrame(results_log)

    if df.empty:
        print("\nTidak ada image yang berhasil dievaluasi.")
        return df

    # =============================================================
    # 6. OVERALL SUMMARY
    # =============================================================
    perfect_pct = (
        df["image_perfect"].mean() * 100
    )

    recall_pct = (
        df["correctly_identified"].sum()
        /
        (df["gt_classes"].sum() + 1e-9)
        * 100
    )

    avg_spurious = df["spurious_classes"].mean()

    print("\n" + "=" * 65)
    print("PER-IMAGE CLASS PRESENCE AUDIT")
    print("=" * 65)

    print(
        f"Total images evaluated       : {len(df)}"
    )

    print(
        f"Perfect images               : "
        f"{df['image_perfect'].sum()} / "
        f"{len(df)} "
        f"({perfect_pct:.1f}%)"
    )

    print(
        f"Overall class recall        : "
        f"{recall_pct:.1f}%"
    )

    print(
        f"Average spurious classes    : "
        f"{avg_spurious:.2f} / image"
    )

    # =============================================================
    # 7. SUMMARY PER SPLIT
    # =============================================================
    print("\n--- Per Split ---")

    for split in df["split"].unique():

        split_df = df[df["split"] == split]

        split_perfect_pct = (
            split_df["image_perfect"].mean() * 100
        )

        split_recall_pct = (
            split_df["correctly_identified"].sum()
            /
            (split_df["gt_classes"].sum() + 1e-9)
            * 100
        )

        split_spurious = (
            split_df["spurious_classes"].mean()
        )

        print(
            f"{split.upper():<8} | "
            f"Images: {len(split_df):<4} | "
            f"Perfect: "
            f"{split_df['image_perfect'].sum():<4} "
            f"({split_perfect_pct:>5.1f}%) | "
            f"Recall: {split_recall_pct:>5.1f}% | "
            f"Spurious: {split_spurious:.2f}"
        )

    return df


# ================================================================
# RUN AUDIT
# ================================================================

per_image_audit_df = evaluate_dataset_per_image_class_presence(
    model=model,
    dataset_dir=DATASET_DIR,
    splits=["train", "valid"],
    conf_thresholds=optimal_thresholds,
    default_conf=0.25,
)


# ================================================================
# SAVE TO NEW CSV
# ================================================================

PER_IMAGE_AUDIT_CSV = (
    EXPERIMENT_DIR
    / "per_image_class_presence_audit_train_valid.csv"
)

per_image_audit_df.to_csv(
    PER_IMAGE_AUDIT_CSV,
    index=False
)

print(
    f"\nDisimpan ke: {PER_IMAGE_AUDIT_CSV}"
)




Evaluating TRAIN: 771 images

Evaluating VALID: 55 images

PER-IMAGE CLASS PRESENCE AUDIT
Total images evaluated       : 826
Perfect images               : 289 / 826 (35.0%)
Overall class recall        : 84.8%
Average spurious classes    : 0.84 / image

--- Per Split ---
TRAIN    | Images: 771  | Perfect: 280  ( 36.3%) | Recall:  85.8% | Spurious: 0.82
VALID    | Images: 55   | Perfect: 9    ( 16.4%) | Recall:  70.9% | Spurious: 1.11

Disimpan ke: D:\PREP_INTERN\nutrivision_pro\runs\v4\segmentation\stage1\per_image_class_presence_audit_train_valid.csv
